## Execute Image Tagging

- In some cases, the model’s response does not follow the expected output format. When this happens, the corresponding image is **skipped**.
- The tagging script is adapted from: https://github.com/google-deepmind/cube/blob/main/cultural_diversity.ipynb
- For each image, the script makes up to **3 attempts** to obtain a valid annotation before giving up and moving on.


In [ ]:
from openai import OpenAI
import os
import tqdm
import os
from dotenv import load_dotenv
from openai import OpenAI 
import base64
import requests

import os
import time
import tqdm
import pandas as pd

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Ref: https://platform.openai.com/docs/guides/images-vision?api-mode=responses
def get_gpt4_response(image_path, question):

    with open(image_path, "rb") as image_file:
      base64_image = base64.b64encode(image_file.read()).decode('utf-8')

    headers = {
      "Content-Type": "application/json",
      "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"
    }

    payload = {
      "model": "gpt-4-turbo",
      "messages": [
        {
          "role": "user",
          "content": [
            {
              "type": "text",
              "text": f"{question}"
            },
            {
              "type": "image_url",
              "image_url": {
                "url": f"data:image/jpeg;base64,{base64_image}"
              }
            }
          ]
        }
      ],
      "max_tokens": 300
    }

    response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload)

    return response.json()['choices'][0]['message']['content']

def tag_images_in_folder(image_paths, start_idx, prompt):
    all_annotations = []
    index = start_idx
    for image_path in tqdm.tqdm(image_paths[start_idx:], initial=start_idx, total=len(image_paths)):
        try:
            annotation = {}
            annotation['image_name'] = os.path.basename(image_path)

            answer = get_gpt4_response(image_path, prompt).strip()
            parts = [x.strip() for x in answer.split(',')]

            if len(parts) != 3:
                print(f"Unexpected format for image {image_path}: {answer}. SKIPPING.")
                continue

            continent, country, artifact = parts

            annotation['label'] = {
                'continent': continent,
                'country': country,
                'artifact': artifact
            }

            all_annotations.append(annotation)
            index += 1
            #time.sleep(1)  

        except Exception as e:
            print(f"Error on image {image_path}: {e}")
          
            return all_annotations, index

    return all_annotations, index


def save_annotations(all_annotations,parent_path, folder_name):

    if not os.path.exists(parent_path):
        os.makedirs(parent_path)
    rows = []

    for ann in all_annotations:
        rows.append({
            'image_name': ann['image_name'],
            'continent': ann['label']['continent'],
            'country': ann['label']['country'],
            'artifact': ann['label']['artifact'],
            'prompt': folder_name.replace("-", " ")
        })

    df = pd.DataFrame(rows, columns=['image_name', 'continent', 'country', 'artifact', 'prompt'])

    save_path = f"{parent_path}/{folder_name}.csv"
    df.to_csv(save_path, index=False)


def process_folder(folder_path, save_path):
    folder_name = os.path.basename(folder_path)
    sample_prompt = folder_name.replace("-", " ")

    # Ref: https://github.com/google-deepmind/cube/blob/main/cultural_diversity.ipynb
    question1 = f"""Given is an image generated for prompt: {sample_prompt}.
                    Which country is the artifact in image from?
                    Which continent does the country belong to?
                    What is the name of the cultural artifact?
                    Output: continent, country, artifact name (comma-separated)"""


    image_paths = [
        os.path.join(root, file)
        for root, _, files in os.walk(folder_path)
        for file in files
        if file.endswith(('.jpg', '.png'))
    ]

    all_annotations = []
    index = 0
    attempts = 3

    while index < len(image_paths) and attempts > 0:
        print(f"Processing from index {index} (attempts left: {attempts})")
        try:
            new_annotations, index = tag_images_in_folder(image_paths, index, question1)
            all_annotations.extend(new_annotations)

            if index < len(image_paths):
                print(f"Retrying from index {index}... Sleeping 5s before retrying.")
                time.sleep(5)
                attempts -= 1  
        except Exception as e:
            attempts -= 1
            print(f" Error occurred: {e}. Retrying... {attempts} attempts left.")
            time.sleep(5)

    if index < len(image_paths):
        print(f"Stopped early at index {index}/{len(image_paths)} after retries exhausted.")
        
        
    save_annotations(all_annotations, save_path, folder_name)
    return all_annotations


In [ ]:
parent_folder = "<folder_path_to_cuisine/landmarks/art>"
# Get all subfolder paths
subfolder_paths = [os.path.join(parent_folder, name) for name in os.listdir(parent_folder) if os.path.isdir(os.path.join(parent_folder, name))]
save_folder = "<save_folder>"

for folder_path in subfolder_paths: 
    process_folder(folder_path, save_folder)